In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import List
import re
import matplotlib.pyplot as plt
import pandas as pd

@dataclass(frozen=True)
class CsvEntry:
    elbow: int          # e.g., 5,10,...
    model: str          # e.g., "EleutherAI/pythia-1B-deduped"
    subset: str         # "common" or "longtail"
    mode: str           # "boost" or "suppress"
    path: Path          # CSV file path

class ZipfPlotter:
    def __init__(self, base_dir: Path, out_dir: Path):
        self.base_dir = Path(base_dir).expanduser().resolve()
        self.out_dir = Path(out_dir).expanduser().resolve()
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.entries: List[CsvEntry] = self.discover_csvs()

    # =========================
    # 1) Discovery & Data Utils
    # =========================
    def discover_csvs(self) -> List[CsvEntry]:
        """
        Compatible with:
          base/elbow_10/.../prob/{common,longtail}/(boost|suppress)/500_all.csv
          base/0_10/...   /prob/{common,longtail}/(boost|suppress)/500_all.csv
        """
        entries: List[CsvEntry] = []

        for csv_path in self.base_dir.rglob("prob/*/*/500_all.csv"):
            try:
                rel = csv_path.relative_to(self.base_dir)
            except ValueError:
                continue
            parts = rel.parts
            if len(parts) < 6:
                continue

            # parse elbow number from top dir
            top_dir = parts[0]
            m = re.search(r'(\d+)(?!.*\d)', top_dir)
            if not m:
                continue
            elbow = int(m.group(1))

            # find 'prob' index
            try:
                prob_idx = parts.index("prob")
            except ValueError:
                continue

            # model spans [1:prob_idx)
            model_parts = parts[1:prob_idx]
            model = "/".join(model_parts) if model_parts else "UNKNOWN_MODEL"

            # subset (common/longtail)
            subset = parts[prob_idx + 1]
            if subset not in {"common", "longtail"}:
                continue

            # mode (boost/suppress)
            mode = parts[prob_idx + 2]
            if mode not in {"boost", "suppress"}:
                continue

            entries.append(CsvEntry(
                elbow=elbow,
                model=model,
                subset=subset,
                mode=mode,
                path=csv_path.resolve()
            ))

        entries.sort(key=lambda e: (e.model, e.subset, e.mode, e.elbow))
        return entries

    def load_csv(self, entry: CsvEntry) -> pd.DataFrame:
        df = pd.read_csv(entry.path)
        if "step" not in df.columns or "prob" not in df.columns:
            raise ValueError(f"CSV missing required columns: {entry.path}")
        return df

    # ===============
    # 2) Plot Methods
    # ===============
    def plot_model_across_elbows(self, model: str, mode: str, step: int):
        subset_entries = [e for e in self.entries if e.model == model and e.mode == mode]
        if not subset_entries:
            print(f"[WARN] No entries for {model}, {mode}")
            return

        subset = subset_entries[0].subset
        fig, ax = plt.subplots(figsize=(6, 4))

        for e in subset_entries:
            df = self.load_csv(e)
            step_df = df[df["step"] == step]
            if step_df.empty:
                continue
            ax.plot(step_df.index, step_df["prob"], label=f"elbow={e.elbow}")

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("rank")
        ax.set_ylabel("prob")
        step_str = str(step)
        ax.set_title(f"Zipf: model={model} | subset={subset} | mode={mode} | step={step_str} | across elbows")
        ax.legend(fontsize="small")
        safe_name = lambda s: s.replace("/", "_")
        fname = f"{safe_name(model)}_{subset}_{mode}_step{step_str}_across_elbows_seq.png"
        plt.tight_layout()
        plt.savefig(self.out_dir / fname, dpi=150)
        plt.close(fig)

    def plot_model_across_steps_at_elbow(self, model: str, mode: str, elbow: int):
        grp = [e for e in self.entries if e.model == model and e.mode == mode and e.elbow == elbow]
        if not grp:
            print(f"[WARN] No entries for {model}, {mode}, elbow={elbow}")
            return

        subset = grp[0].subset
        fig, ax = plt.subplots(figsize=(6, 4))

        for e in grp:
            df = self.load_csv(e)
            for step in sorted(df["step"].unique()):
                step_df = df[df["step"] == step]
                ax.plot(step_df.index, step_df["prob"], label=f"step={step}")

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("rank")
        ax.set_ylabel("prob")
        ax.set_title(f"Zipf: model={model} | subset={subset} | mode={mode} | elbow={elbow} | across steps")
        ax.legend(fontsize="small")
        safe_name = lambda s: s.replace("/", "_")
        fname = f"{safe_name(model)}_{subset}_{mode}_elbow{elbow}_across_steps_seq.png"
        plt.tight_layout()
        plt.savefig(self.out_dir / fname, dpi=150)
        plt.close(fig)

    # ====================
    # 3) Batch Generators
    # ====================
    def batch_plot_elbows(self, step: int = 1000):
        for model in sorted(set(e.model for e in self.entries)):
            for mode in ["boost", "suppress"]:
                for subset in ["common", "longtail"]:
                    subset_entries = [e for e in self.entries if e.model == model and e.mode == mode and e.subset == subset]
                    if subset_entries:
                        self.plot_model_across_elbows(model, mode, step)

    def batch_plot_steps(self, elbow: int = 10):
        for model in sorted(set(e.model for e in self.entries)):
            for mode in ["boost", "suppress"]:
                for subset in ["common", "longtail"]:
                    subset_entries = [e for e in self.entries if e.model == model and e.mode == mode and e.subset == subset and e.elbow == elbow]
                    if subset_entries:
                        self.plot_model_across_steps_at_elbow(model, mode, elbow)

    def batch_plot_all(self):
        """
        For each (model, subset, mode):
          - plot across elbows for every step found in the CSVs
          - plot across steps for every elbow found in the CSVs
        """
        combos = {(e.model, e.subset, e.mode) for e in self.entries}
        for model, subset, mode in combos:
            subset_entries = [e for e in self.entries if e.model == model and e.subset == subset and e.mode == mode]

            # all steps
            all_steps = sorted({s for e in subset_entries for s in self.load_csv(e)["step"].unique()})
            for step in all_steps:
                self.plot_model_across_elbows(model=model, mode=mode, step=step)

            # all elbows
            all_elbows = sorted({e.elbow for e in subset_entries})
            for elbow in all_elbows:
                self.plot_model_across_steps_at_elbow(model=model, mode=mode, elbow=elbow)


In [2]:
zp = ZipfPlotter(base_dir="/Users/jliu/workspace/RAG/results/selection/neuron",
                 out_dir="/Users/jliu/workspace/RAG/results/selection/fig")

zp.batch_plot_all()


ValueError: CSV missing required columns: /Users/jliu/workspace/RAG/results/selection/neuron/15_elbow/gpt2-xl/prob/common/boost/500_all.csv